# imports

In [ ]:
!pip install pyarabic scikit-learn networkx numpy nltk rouge-score -q
!pip install stanza transformers torch -q

In [ ]:
# -*- coding: utf-8 -*-
"""
Arabic Legal Case Pipeline — ALL-IN-ONE (Colab / Jupyter)
=============================================================
ملف واحد يجمع كل شي: الـ pipeline + التقييم + مساعد بناء الـ gold data.
لا يحتاج أي import محلي (كل الاعتماديات مكتبات عامة فقط) -- انسخو
كامل بخلية وحدة بـ Colab/Jupyter وشغّلوها.

المكتبات المطلوبة (خلية أولى قبل هاد الكود):
    !pip install pyarabic scikit-learn networkx numpy nltk rouge-score -q

الأقسام:
  [1] Preprocessing        — UniversalPreprocessor
  [2] Extractive Summarizer— LegalSummarizer (TextRank + MMR + cue phrases)
  [3] Structured Extractor — StructuredFieldExtractor (regex فقط)
  [4] Fact Extractor (اختياري) — Stanza dependency parsing
  [5] Entity Extractor (اختياري) — HF NER model
  [6] Pipeline النهائي     — IntelligentLegalPipeline
  [7] Evaluation           — ROUGE + sentence-selection F1 + field F1
  [8] Gold Data Builder    — annotate_case_interactive() / build_gold_data()

كيفية الاستخدام السريع بـ Colab:
    pipeline = IntelligentLegalPipeline(enable_fact_extraction=False, enable_ner=False)
    result = pipeline.analyze("النص هون...")

    # لبناء الـ 50 حالة تفاعلياً:
    build_gold_data(cases=[{"text": "..."}, ...], output_path="gold_data.json", n=50)

    # للتقييم بعد ما تخلص الـ gold data:
    gold_data = load_gold_data("gold_data.json")
    report = run_evaluation(gold_data, pipeline)
    print(json.dumps(report, ensure_ascii=False, indent=2))
"""

import re
import os
import json
from collections import Counter

import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from rouge_score import rouge_scorer

try:
    import pyarabic.araby as araby
    HAS_PYARABIC = True
except ImportError:
    HAS_PYARABIC = False

try:
    import nltk
    from nltk.corpus import stopwords as nltk_stopwords
    try:
        _ = nltk_stopwords.words("arabic")
        HAS_NLTK_STOPWORDS = True
    except LookupError:
        try:
            nltk.download("stopwords", quiet=True)
            _ = nltk_stopwords.words("arabic")
            HAS_NLTK_STOPWORDS = True
        except Exception:
            HAS_NLTK_STOPWORDS = False
except ImportError:
    HAS_NLTK_STOPWORDS = False


# 1- preprocessing

In [ ]:
# =============================================================================
# [1] PREPROCESSOR
# =============================================================================

LEGAL_STOPWORDS = {
    "محكمة", "المحكمة", "قرار", "رقم", "تاريخ", "القاضي",
    "باسم", "الشعب", "العربي", "السوري", "قانون", "مادة",
    "بناء", "عليه", "حيث", "ان", "إن", "لذلك", "قررت",
    "الدعوى", "الأساس", "الغرفة", "الجزائية", "المدنية",
}

GENERAL_STOPWORDS_FALLBACK = {
    "في", "من", "إلى", "على", "عن", "مع", "هذا", "هذه", "ذلك", "التي", "الذي",
    "و", "أو", "ثم", "كان", "كانت", "يكون", "أن", "لا", "ما", "لم", "لن",
    "قد", "بعد", "قبل", "عند", "كل", "بعض", "غير", "بين", "حتى", "إذا", "كما",
    "له", "لها", "لهم", "به", "بها", "بهم", "هو", "هي", "هم", "أنا", "نحن",
}


class UniversalPreprocessor:
    def __init__(self):
        general_sw = set(nltk_stopwords.words("arabic")) if HAS_NLTK_STOPWORDS else GENERAL_STOPWORDS_FALLBACK
        self.all_stopwords = list(general_sw.union(LEGAL_STOPWORDS))

    def clean_text(self, text_input: str) -> str:
        if not isinstance(text_input, str):
            return ""
        text = re.sub(r"[\u064B-\u065F\u0670]", "", text_input)  # التشكيل
        text = re.sub(r"ـ", "", text)                             # التطويل
        text = re.sub(r"[أإآ]", "ا", text)                        # الألف
        text = re.sub(r"ة", "ه", text)                            # التاء المربوطة
        text = re.sub(r"ى", "ي", text)                            # الألف المقصورة
        text = re.sub(r"ؤ", "ء", text)
        text = re.sub(r"ئ", "ء", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def remove_stopwords(self, text: str) -> str:
        words = text.split()
        filtered = [w for w in words if w not in self.all_stopwords]
        return " ".join(filtered)

    def tokenize(self, text: str):
        if HAS_PYARABIC:
            return araby.tokenize(text)
        return text.split()



# 2- legal summarizer

In [ ]:

# =============================================================================
# [2] LEGAL SUMMARIZER (TextRank + MMR + ترجيح كلمات حساسة قانونياً)
# =============================================================================

CUE_PHRASE_CATEGORIES = {
    "confession_denial": (r"اعترف|انكر|نفي|طعن|اسقط|اقر|ادعي", 1.6),
    "arrest_action":     (r"القت القبض|القي القبض|تم توقيف|داهمت|كمشت|قبضت", 1.5),
    "ruling":            (r"قرر القاضي|حكمت المحكمه|قررت المحكمه|بناء علي ما تقدم|قرر توقيف|الزمت المحكمه", 1.7),
    "roles":             (r"المتهم|المجني عليه|الشاهد|الشهود|المشتكي|المدعي|المدعي عليه", 1.2),
    "evidence":          (r"دليل|اداه|سلاح|بصمات|كاميرا|شهاده|سند", 1.3),
    "date":              (r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", 1.4),
}


class LegalSummarizer:
    def __init__(self, preprocessor: UniversalPreprocessor = None):
        self.preprocessor = preprocessor or UniversalPreprocessor()

    def split_into_sentences(self, text: str):
        """نقطة/؟/! أو سطر جديد فقط. لا نقسّم على الفاصلة «،» لأنها غالباً
        داخل الجملة نفسها بالعربي الفصيح -- القسمة عليها كانت تقطع جملة
        الحكم/القرار عن باقيها وتفقدها من الملخص."""
        sentences = re.split(r"(?<=[.؟!])\s+|\n+", text.strip())
        return [s.strip() for s in sentences if len(s.strip()) > 10]

    def _cue_score(self, clean_sentence: str) -> float:
        score = 1.0
        for _, (pattern, weight) in CUE_PHRASE_CATEGORIES.items():
            if re.search(pattern, clean_sentence):
                score *= weight
        return score

    def summarize(self, text, base_compression_ratio=0.5, max_sentences=10,
                  min_sentences_for_compression=7, use_mmr=True, mmr_lambda=0.7):
        summary_text, _, _ = self.summarize_with_indices(
            text, base_compression_ratio, max_sentences,
            min_sentences_for_compression, use_mmr, mmr_lambda,
        )
        return summary_text

    def summarize_with_indices(self, text, base_compression_ratio=0.5, max_sentences=10,
                                min_sentences_for_compression=7, use_mmr=True, mmr_lambda=0.7):
        """متل summarize()، بس كمان بترجع مؤشرات الجمل المختارة -- لازمة
        للتقييم على مستوى الجملة (sentence-selection F1)."""
        sentences = self.split_into_sentences(text)
        total_sentences = len(sentences)

        if total_sentences <= min_sentences_for_compression:
            return " ".join(sentences), list(range(total_sentences)), sentences

        target_length = int(total_sentences * base_compression_ratio)
        target_length = min(target_length, max_sentences)
        target_length = max(target_length, 4)

        clean_sentences = [self.preprocessor.clean_text(s) for s in sentences]

        vectorizer = TfidfVectorizer(stop_words=self.preprocessor.all_stopwords)
        X = vectorizer.fit_transform(clean_sentences)

        sim_matrix = (X * X.T).toarray()
        np.fill_diagonal(sim_matrix, 0)
        nx_graph = nx.from_numpy_array(sim_matrix)
        try:
            scores = nx.pagerank(nx_graph)
        except nx.PowerIterationFailedConvergence:
            scores = {i: 1.0 / total_sentences for i in range(total_sentences)}

        for i, clean_sentence in enumerate(clean_sentences):
            scores[i] *= self._cue_score(clean_sentence)

        if use_mmr:
            selected = self._mmr_select(scores, sim_matrix, target_length, mmr_lambda)
        else:
            ranked = sorted(((scores[i], i) for i in range(total_sentences)), reverse=True)
            selected = [i for _, i in ranked[:target_length]]

        selected = sorted(selected)
        summary_text = " ".join(sentences[i] for i in selected)
        return summary_text, selected, sentences

    def _mmr_select(self, scores, sim_matrix, top_k, lambda_param):
        n = len(scores)
        top_k = min(top_k, n)
        selected, candidates = [], list(range(n))

        while len(selected) < top_k and candidates:
            if not selected:
                best = max(candidates, key=lambda i: scores[i])
            else:
                def mmr_score(i):
                    redundancy = max(sim_matrix[i][j] for j in selected)
                    return lambda_param * scores[i] - (1 - lambda_param) * redundancy
                best = max(candidates, key=mmr_score)
            selected.append(best)
            candidates.remove(best)
        return selected



# 3- STRUCTURED FIELD EXTRACTOR

In [ ]:

# =============================================================================
# [3] STRUCTURED FIELD EXTRACTOR (regex بحت -> صفر اختلاق)
# =============================================================================

class StructuredFieldExtractor:
    def extract(self, text: str) -> dict:
        fields = {
            "date": None, "accused": [], "victim": None, "action": None,
            "evidence": [], "confession_status": [], "ruling": None,
        }

        date_match = re.search(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", text)
        if date_match:
            fields["date"] = date_match.group()

        # "المتهم" بالقضايا الجزائية، "المدعى عليه" بالقضايا المدنية/الاحتيال
        party_pattern = r"(?:المتهم(?:\s+(?:الأول|الثاني|الثالث))?|المدعى عليه)"

        fields["accused"] = list(dict.fromkeys(re.findall(party_pattern, text)))

        if re.search(r"المجني عليه", text):
            fields["victim"] = "المجني عليه"
        elif re.search(r"المدعي(?!\s*عليه)", text):
            fields["victim"] = "المدعي"

        seen = set()
        for pattern in [
            rf"(اعترف|أنكر|انكر)[^\.]{{0,40}}?({party_pattern})",
            rf"({party_pattern})[^\.]{{0,40}}?(اعترف|أنكر|انكر)",
        ]:
            for m in re.finditer(pattern, text):
                g1, g2 = m.group(1), m.group(2)
                who, verb = (g2, g1) if g1 in ("اعترف", "أنكر", "انكر") else (g1, g2)
                if (who, verb) not in seen:
                    seen.add((who, verb))
                    fields["confession_status"].append(f"{who}: {verb}")

        fields["evidence"] = list(dict.fromkeys(
            re.findall(r"أداة\s+[^\s.،؛!؟]+|سلاح\s*[^\s.،؛!؟]*|بصمات|كاميرا|سند\s+[^\s.،؛!؟]+", text)
        ))

        ruling_match = re.search(
            r"(قرر القاضي[^\.]*\.)|(حكمت المحكمة[^\.]*\.)|(قررت المحكمة[^\.]*\.)|(ألزمت المحكمة[^\.]*\.)",
            text,
        )
        if ruling_match:
            fields["ruling"] = ruling_match.group().strip()

        action_match = re.search(
            r"(سرق[^\.]*\.|ضرب[^\.]*\.|مشاجرة[^\.]*\.|اعتدى[^\.]*\.|احتيال[^\.]*\.)", text
        )
        if action_match:
            fields["action"] = action_match.group().strip()

        return fields




# 4- FACT EXTRACTOR

In [ ]:
# =============================================================================
# [4] FACT EXTRACTOR (اختياري) — Stanza dependency parsing
# =============================================================================

class FactExtractor:
    """محتاج: !pip install stanza -q  (بينزل نموذج 'ar' أول مرة)."""

    def __init__(self, use_gpu=False):
        self._nlp = None
        self._available = False
        try:
            import stanza
            stanza.download("ar", verbose=False)
            self._nlp = stanza.Pipeline("ar", processors="tokenize,mwt,pos,lemma,depparse",
                                         use_gpu=use_gpu, verbose=False)
            self._available = True
        except Exception as e:
            print(f"⚠️ FactExtractor غير متوفر (Stanza فشل بالتحميل): {e}")

    @property
    def available(self):
        return self._available

    def extract_facts(self, text: str):
        if not self._available or not text or len(text.strip()) < 5:
            return []

        doc = self._nlp(text)
        facts = []
        for sentence in doc.sentences:
            verbs = [w for w in sentence.words if w.upos == "VERB"]
            for verb in verbs:
                subject_text = "مستتر/محذوف"
                objects = []
                for word in sentence.words:
                    if word.head == verb.id:
                        if word.deprel in ["nsubj", "nsubj:pass", "csubj"]:
                            subject_text = word.text
                        elif word.deprel in ["obj", "iobj", "obl", "obl:arg", "xcomp", "ccomp"]:
                            objects.append(word.text)
                object_text = " ".join(objects) if objects else "غير محدد"
                if subject_text != "مستتر/محذوف" or object_text != "غير محدد":
                    facts.append(f"({subject_text} ➔ {verb.lemma} ➔ {object_text})")
        return facts



# 5- ENTITY EXTRACTOR

In [ ]:
# =============================================================================
# [5] ENTITY EXTRACTOR (اختياري) — HF NER model + regex
# =============================================================================

class EntityExtractor:
    """محتاج: !pip install transformers torch -q (بينزل نموذج hatmimoha/arabic-ner)."""

    def __init__(self):
        self._ner_pipeline = None
        self._available = False
        try:
            from transformers import pipeline as hf_pipeline
            self._ner_pipeline = hf_pipeline(
                "ner", model="hatmimoha/arabic-ner", aggregation_strategy="simple"
            )
            self._available = True
        except Exception as e:
            print(f"⚠️ EntityExtractor غير متوفر (تحميل نموذج NER فشل): {e}")

        self.date_pattern = r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"
        self.money_pattern = (
            r"(?:\d+\s*(?:مليون|ألف|ليرة|دولار|يورو))"
            r"|(?:(?:مبلغاً وقدره|مبلغ|قدره)\s+([أ-ي\s]+(?:ليرة|سورية|دولار|يورو)))"
        )
        self.charge_pattern = r"(?:بجرم|تهمة|بجناية|بجنحة)\s+([أ-ي]+(?:\s+(?!و|في|على|من|إلى)[أ-ي]+)?)"

    @property
    def available(self):
        return self._available

    def extract_entities(self, text: str) -> dict:
        entities = {
            "الأشخاص": set(), "الأماكن": set(), "المنظمات": set(),
            "التواريخ": set(), "المبالغ_المالية": set(), "التهم_والجرائم": set(),
        }
        if not text or len(text.strip()) < 2:
            return {}

        if self._available:
            ai_results = self._ner_pipeline(text)
            for ent in ai_results:
                start, end = ent.get("start"), ent.get("end")
                if start is not None and end is not None:
                    true_start = text.rfind(" ", 0, start)
                    true_start = true_start + 1 if true_start != -1 else 0
                    true_end = text.find(" ", end)
                    true_end = true_end if true_end != -1 else len(text)
                    word = text[true_start:true_end].strip()
                    word = re.sub(r"^[،.؛!؟,]+|[،.؛!؟,]+$", "", word).strip()
                else:
                    word = ent.get("word", "").replace("##", "").strip()

                if len(word) <= 1:
                    continue

                group = ent.get("entity_group", "")
                if group in ["PER", "PERSON"]:
                    entities["الأشخاص"].add(word)
                elif group in ["LOC", "LOCATION"]:
                    entities["الأماكن"].add(word)
                elif group in ["ORG", "ORGANIZATION"]:
                    entities["المنظمات"].add(word)

        entities["التواريخ"].update(re.findall(self.date_pattern, text))
        for m in re.finditer(self.money_pattern, text):
            entities["المبالغ_المالية"].add((m.group(1) or m.group(0)).strip())
        for m in re.finditer(self.charge_pattern, text):
            entities["التهم_والجرائم"].add(m.group(1).strip())

        return {k: list(v) for k, v in entities.items() if v}



# 6- pipeline

In [ ]:
# =============================================================================
# [6] PIPELINE النهائي
# =============================================================================

class IntelligentLegalPipeline:
    def __init__(self, enable_fact_extraction=True, enable_ner=True):
        print("⚙️ جاري بناء الـ pipeline...")
        self.preprocessor = UniversalPreprocessor()
        self.summarizer = LegalSummarizer(self.preprocessor)
        self.field_extractor = StructuredFieldExtractor()

        self.fact_extractor = FactExtractor() if enable_fact_extraction else None
        self.entity_extractor = EntityExtractor() if enable_ner else None
        print("✅ الـ pipeline جاهز.\n")

    def analyze(self, raw_text: str) -> dict:
        if not raw_text or len(raw_text.strip()) < 10:
            return {"status": "error", "error": "النص المدخل قصير جداً ولا يمكن تحليله."}

        clean_text = self.preprocessor.clean_text(raw_text)
        summary = self.summarizer.summarize(raw_text)
        structured_fields = self.field_extractor.extract(raw_text)

        facts = []
        if self.fact_extractor and self.fact_extractor.available:
            facts = self.fact_extractor.extract_facts(summary)

        entities = {}
        if self.entity_extractor and self.entity_extractor.available:
            entities = self.entity_extractor.extract_entities(clean_text)
        else:
            fallback_extractor = EntityExtractor.__new__(EntityExtractor)
            fallback_extractor._available = False
            fallback_extractor.date_pattern = r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"
            fallback_extractor.money_pattern = (
                r"(?:\d+\s*(?:مليون|ألف|ليرة|دولار|يورو))"
                r"|(?:(?:مبلغاً وقدره|مبلغ|قدره)\s+([أ-ي\s]+(?:ليرة|سورية|دولار|يورو)))"
            )
            fallback_extractor.charge_pattern = r"(?:بجرم|تهمة|بجناية|بجنحة)\s+([أ-ي]+(?:\s+(?!و|في|على|من|إلى)[أ-ي]+)?)"
            entities = fallback_extractor.extract_entities(clean_text)

        return {
            "status": "success",
            "original_length": len(raw_text),
            "analysis": {
                "extractive_summary": summary,
                "structured_fields": structured_fields,
                "facts_triples": facts,
                "entities": entities,
            },
        }



# 7- Evaluation

In [ ]:

# =============================================================================
# [7] EVALUATION
# =============================================================================

_preprocessor_eval = UniversalPreprocessor()
_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)


def normalize_for_rouge(text: str) -> str:
    return _preprocessor_eval.clean_text(text)


def compute_rouge(predicted_summary: str, gold_summary: str) -> dict:
    pred_norm = normalize_for_rouge(predicted_summary)
    gold_norm = normalize_for_rouge(gold_summary)
    scores = _rouge.score(gold_norm, pred_norm)
    return {
        "rouge1_f": round(scores["rouge1"].fmeasure, 3),
        "rouge2_f": round(scores["rouge2"].fmeasure, 3),
        "rougeL_f": round(scores["rougeL"].fmeasure, 3),
    }


def _normalize_token_set(items):
    if items is None:
        return set()
    if isinstance(items, str):
        items = [items]
    return {normalize_for_rouge(x) for x in items if x}


def _prf1(gold_set, pred_set):
    if not gold_set and not pred_set:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0}
    if not pred_set or not gold_set:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    tp = len(gold_set & pred_set)
    precision = tp / len(pred_set)
    recall = tp / len(gold_set)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)}


def sentence_selection_prf1(gold_indices, predicted_indices):
    """يقارن مباشرة: هل الجمل يلي اختارها الإنسان هي نفسها يلي اختارها الـ
    pipeline؟ المقياس الأدق لأسلوب gold الاستخلاصي (اختيار جمل)."""
    return _prf1(set(gold_indices), set(predicted_indices))


def _token_f1(gold_text, pred_text):
    gold_tokens = normalize_for_rouge(gold_text or "").split()
    pred_tokens = normalize_for_rouge(pred_text or "").split()
    if not gold_tokens and not pred_tokens:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0}
    if not gold_tokens or not pred_tokens:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    common = Counter(gold_tokens) & Counter(pred_tokens)
    tp = sum(common.values())
    if tp == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    precision = tp / len(pred_tokens)
    recall = tp / len(gold_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)}


SET_FIELDS = ["accused", "evidence", "confession_status"]
SCALAR_FIELDS = ["date", "victim"]
FREE_TEXT_FIELDS = ["action", "ruling"]


def evaluate_fields(gold_fields: dict, predicted_fields: dict) -> dict:
    result = {}
    for field in SET_FIELDS:
        result[field] = _prf1(_normalize_token_set(gold_fields.get(field)),
                               _normalize_token_set(predicted_fields.get(field)))
    for field in SCALAR_FIELDS:
        gold_val = normalize_for_rouge(gold_fields.get(field) or "")
        pred_val = normalize_for_rouge(predicted_fields.get(field) or "")
        match = 1.0 if gold_val and gold_val == pred_val else (1.0 if not gold_val and not pred_val else 0.0)
        result[field] = {"exact_match": match}
    for field in FREE_TEXT_FIELDS:
        result[field] = _token_f1(gold_fields.get(field), predicted_fields.get(field))
    return result


def lead_k_baseline(text: str, k: int = 3, summarizer: LegalSummarizer = None) -> str:
    summarizer = summarizer or LegalSummarizer()
    sentences = summarizer.split_into_sentences(text)
    return " ".join(sentences[:k])


def aggregate(list_of_dicts, keys):
    agg = {}
    for key in keys:
        vals = [d[key] for d in list_of_dicts if key in d]
        if vals and isinstance(vals[0], dict):
            agg[key] = {metric: round(sum(v[metric] for v in vals) / len(vals), 3) for metric in vals[0]}
        else:
            agg[key] = round(sum(vals) / len(vals), 3) if vals else None
    return agg


def run_evaluation(gold_data, pipeline: "IntelligentLegalPipeline"):
    field_extractor = StructuredFieldExtractor()
    rouge_results, rouge_baseline_results, field_results = [], [], []
    sentence_selection_results = []
    has_sentence_indices = all("gold_sentence_indices" in c for c in gold_data)

    for case in gold_data:
        text = case["text"]
        gold_summary = case["gold_summary"]
        gold_fields = case["gold_fields"]

        predicted_summary, predicted_indices, _ = pipeline.summarizer.summarize_with_indices(text)
        predicted_fields = field_extractor.extract(text)

        rouge_results.append(compute_rouge(predicted_summary, gold_summary))
        field_results.append(evaluate_fields(gold_fields, predicted_fields))

        if has_sentence_indices:
            sentence_selection_results.append(
                sentence_selection_prf1(case["gold_sentence_indices"], predicted_indices)
            )

        baseline_summary = lead_k_baseline(text, k=3, summarizer=pipeline.summarizer)
        rouge_baseline_results.append(compute_rouge(baseline_summary, gold_summary))

    report = {
        "n_cases": len(gold_data),
        "track_A_rouge_pipeline": aggregate(rouge_results, ["rouge1_f", "rouge2_f", "rougeL_f"]),
        "track_A_rouge_lead3_baseline": aggregate(rouge_baseline_results, ["rouge1_f", "rouge2_f", "rougeL_f"]),
        "track_B_field_extraction": aggregate(field_results, SET_FIELDS + SCALAR_FIELDS + FREE_TEXT_FIELDS),
    }
    if has_sentence_indices:
        report["track_A_sentence_selection"] = aggregate(
            [{"sentence_selection": s} for s in sentence_selection_results], ["sentence_selection"]
        )
    return report



# 8- GOLD DATA BUILDER

In [ ]:

# =============================================================================
# [8] GOLD DATA BUILDER — تفاعلي، يشتغل مباشرة بخلية Colab/Jupyter
# =============================================================================

def load_existing_gold(output_path):
    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_gold(gold_data, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(gold_data, f, ensure_ascii=False, indent=2)


def load_gold_data(path="gold_data.json"):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"📂 تم تحميل {len(data)} حالة حقيقية من {path}")
        return data
    print(f"⚠️ {path} مش موجود -- عم نستخدم بيانات الديمو (استبدلها ببياناتك الحقيقية)")
    return GOLD_DATA_DEMO


def _prompt_field(label):
    val = input(f"    {label} (Enter للتخطي): ").strip()
    return val if val else None


def _prompt_list_field(label):
    val = input(f"    {label} (افصل بفاصلة، Enter للتخطي): ").strip()
    if not val:
        return []
    return [v.strip() for v in val.split(",") if v.strip()]


def annotate_case_interactive(case_idx, text, summarizer):
    """بيعرض الحالة مقطّعة لجمل مرقّمة، وبياخذ منك أرقام جمل الملخص + الحقول.
    شغّلها بخلية Colab/Jupyter لحالها -- input() بيشتغل عادي جواتها."""
    sentences = summarizer.split_into_sentences(text)

    print(f"\n{'='*70}\nالحالة #{case_idx}\n{'='*70}")
    print(text.strip())
    print("-" * 70)
    print("الجمل مرقّمة:")
    for i, s in enumerate(sentences):
        print(f"  [{i}] {s}")
    print("-" * 70)

    raw = input("أرقام جمل الملخص (مثال: 0,2,4) أو 's' تخطي أو 'q' خروج: ").strip()
    if raw.lower() == "q":
        return "quit"
    if raw.lower() == "s" or not raw:
        return None

    try:
        indices = sorted({int(x.strip()) for x in raw.split(",") if x.strip() != ""})
        indices = [i for i in indices if 0 <= i < len(sentences)]
    except ValueError:
        print("⚠️ إدخال غير صحيح.")
        return None

    gold_summary = " ".join(sentences[i] for i in indices)
    print(f"  -> الملخص المُشكَّل: {gold_summary}")

    print("  عبّي الحقول (أو Enter تخطي كل وحدة):")
    gold_fields = {
        "date": _prompt_field("التاريخ"),
        "accused": _prompt_list_field("المتهم/المدعى عليه (list)"),
        "victim": _prompt_field("المجني عليه/المدعي"),
        "action": _prompt_field("الفعل/الجرم (نص حر)"),
        "evidence": _prompt_list_field("الأدلة (list)"),
        "confession_status": _prompt_list_field('حالة الاعتراف/الإنكار (مثال: "المتهم الأول: أنكر")'),
        "ruling": _prompt_field("القرار/الحكم (نص حر)"),
    }

    return {
        "case_id": case_idx,
        "text": text,
        "gold_sentence_indices": indices,
        "gold_summary": gold_summary,
        "gold_fields": gold_fields,
    }


def build_gold_data(cases, output_path="gold_data.json", n=50):
    """cases: list بالشكل [{"text": "..."}, ...] أو list of strings.
    بتنده هاي الدالة مباشرة بخلية Colab -- ما في حاجة لـ argparse أو سطر أوامر.

    مثال:
        cases = [{"text": t} for t in df["text"].tolist()]   # لو عندك DataFrame
        build_gold_data(cases, output_path="gold_data.json", n=50)
    """
    all_texts = [c["text"] if isinstance(c, dict) else c for c in cases]
    gold_data = load_existing_gold(output_path)
    already_done_ids = {g["case_id"] for g in gold_data}

    print(f"📂 عدد الحالات: {len(all_texts)}")
    print(f"✅ موسوم مسبقاً: {len(already_done_ids)}")
    print(f"🎯 الهدف: {n} حالة\n")

    summarizer = LegalSummarizer(UniversalPreprocessor())

    for idx, text in enumerate(all_texts):
        if len(gold_data) >= n:
            break
        if idx in already_done_ids:
            continue

        result = annotate_case_interactive(idx, text, summarizer)
        if result == "quit":
            break
        if result:
            gold_data.append(result)
            save_gold(gold_data, output_path)  # حفظ فوري بعد كل حالة

    save_gold(gold_data, output_path)
    print(f"\n✅ تم حفظ {len(gold_data)} حالة موسومة بـ: {output_path}")
    return gold_data



# Testing

In [ ]:
# =============================================================================
# DEMO GOLD DATA (استبدلها ببياناتك الحقيقية عبر build_gold_data)
# =============================================================================

_case1_text = "الزلمة فات عالدكانة وسرق المصاري من الدرج. أنا شفته بعيني عم يركض بالشارع وبعدين تخبى. الشرطة كمشته المسا واعترف بكل شي."
_case1_indices = [0, 2]

_case2_text = "بتاريخ 20/05/2023، ورد إخبار إلى قسم الشرطة يفيد بوقوع مشاجرة جماعية في الساحة. وتوجهت الدوريات فوراً إلى المكان حيث ألقت القبض على المتورطين. وأفاد الشهود أن المتهم الأول بادر بضرب المجني عليه باستخدام أداة حادة. وأثناء التحقيق، أنكر المتهم الأول التهمة المنسوبة إليه. من جهة أخرى، اعترف المتهم الثاني بمشاركته في الشجار وتكسير الواجهة. وبناءً على ما تقدم، قرر القاضي توقيف المتهمين ومصادرة الأداة المستخدمة."
_case2_indices = [0, 2, 3, 4, 5]


def _build_demo_entry(text, indices, gold_fields):
    summarizer = LegalSummarizer()
    sentences = summarizer.split_into_sentences(text)
    gold_summary = " ".join(sentences[i] for i in indices)
    return {
        "text": text,
        "gold_sentence_indices": indices,
        "gold_summary": gold_summary,
        "gold_fields": gold_fields,
    }


GOLD_DATA_DEMO = [
    _build_demo_entry(_case1_text, _case1_indices, {
        "date": None, "accused": [], "victim": None,
        "action": "سرق المصاري من الدرج", "evidence": [],
        "confession_status": ["اعترف"], "ruling": None,
    }),
    _build_demo_entry(_case2_text, _case2_indices, {
        "date": "20/05/2023",
        "accused": ["المتهم الأول", "المتهم الثاني"],
        "victim": "المجني عليه",
        "action": "مشاجرة جماعية في الساحة",
        "evidence": ["أداة حادة"],
        "confession_status": ["المتهم الأول: أنكر", "المتهم الثاني: اعترف"],
        "ruling": "قرر القاضي توقيف المتهمين ومصادرة الأداة المستخدمة",
    }),
]



In [ ]:
pipeline = IntelligentLegalPipeline(enable_fact_extraction=False, enable_ner=False)
    gold_data = load_gold_data()  # بيرجع للديمو إذا ما في gold_data.json بعد
    report = run_evaluation(gold_data, pipeline)
    print("\n" + "=" * 70)
    print("EVALUATION REPORT")
    print("=" * 70)
    print(json.dumps(report, ensure_ascii=False, indent=2))

IndentationError: unexpected indent (850470928.py, line 2)

In [ ]:
pipeline = IntelligentLegalPipeline(enable_fact_extraction=False, enable_ner=False)
result = pipeline.analyze("النص هون...")

# لبناء الـ 50 حالة تفاعلياً (input() بيشتغل عادي بخلايا Colab/Jupyter):
cases = [{"text": t} for t in your_dataframe["text"].tolist()]
build_gold_data(cases, output_path="gold_data.json", n=50)

# للتقييم بعدين:
gold_data = load_gold_data("gold_data.json")
report = run_evaluation(gold_data, pipeline)

# MY TEST

In [ ]:
import pandas as pd
import json

# 1. قراءة ملف البيانات
df = pd.read_csv("cases_and_summerization.csv")

# 2. تنظيف الحقول من القيم الفارغة (NaN) لتجنب الأخطاء عند الدمج
df['facts'] = df['facts'].fillna('')
df['trail'] = df['trail'].fillna('')

# 3. دمج حقل الوقائع (facts) وحقل المحاكمة (trail) في فقرة واحدة
# استخدمنا مسافة أو سطر جديد بينهما لضمان عدم تداخل الكلمات
df['merged_text'] = df['facts'].astype(str) + "\n\n" + df['trail'].astype(str)

# 4. تحويل النصوص المدمجة إلى الشكل الذي تقبله دالة بناء البيانات (قائمة من القواميس)
# سيتم تجاهل أي أعمدة أخرى مثل رقم القضية أو اسم الملف تلقائياً لأننا لم نقم بتضمينها هنا
cases = [{"text": text} for text in df['merged_text'].tolist() if len(text.strip()) > 10]

# 5. تهيئة الـ Pipeline
pipeline = IntelligentLegalPipeline(enable_fact_extraction=False, enable_ner=False)

# 6. بناء بيانات التقييم (Gold Data) التفاعلية
# سيقوم الكود بعرض 50 حالة لتضيفي لها الملخص والحقول يدوياً (يمكنك تغيير الرقم n)
print("=== بدء بناء بيانات التقييم ===")
build_gold_data(cases, output_path="gold_data.json", n=50)

# 7. تشغيل التقييم وعرض النتائج بعد الانتهاء من بناء الـ Gold Data
print("\n=== جاري حساب نتائج التقييم ===")
gold_data = load_gold_data("gold_data.json")
report = run_evaluation(gold_data, pipeline)

# طباعة التقرير النهائي
print("\n" + "=" * 70)
print("EVALUATION REPORT")
print("=" * 70)
print(json.dumps(report, ensure_ascii=False, indent=2))